<a href="https://colab.research.google.com/github/zyf-hitsz/PytorchLearning/blob/main/%E7%A5%9E%E7%BB%8F%E7%BD%91%E7%BB%9C/Dropout%E5%B1%82/DropoutLayers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Dropout层
训练时随机关闭一些神经元，防止过拟合。



##数学原理
假设输入：
$$
x=(x_1,x_2,\dots,x_n)
$$定义一个随机 mask：
$$
m_i\sim Bernoulli(1-p)
$$也就是：
$$
m_i=
\begin{cases}
1,&概率1-p\\
0,&概率p
\end{cases}
$$然后：
$$
x_i'=m_ix_i
$$但是 PyTorch 实际采用的是 Inverted Dropout：
$$
\boxed{
x_i'=\frac{m_i}{1-p}x_i
}
$$]这样可以保持期望不变：
$
E[x_i']
=
E\left[\frac{m_i}{1-p}x_i\right]
$而：
$
E[m_i]=1-p
$因此：
$$
E[x_i']
=
\frac{1-p}{1-p}x_i
=x_i
$$也就是说：
$$
\boxed{E[x']=x}
$$为了防止部分数据归零后数据尺度发生明显改变，这样的补偿时很重要的。

dropout仅仅是在训练时防止过拟合，所以在训练模式下dropout会开启，测试模式下会自动关闭。

一般p不会超过0.5，有些场景下p很小甚至为0。

In [5]:
import torch
import torch.nn as nn
input = torch.randn(2,3)
dropout = nn.Dropout(p=0.5)
output = dropout(input)
print(f"input:{input}")
print(f"ouput:{output}")

input:tensor([[-1.1297,  0.4487, -1.1150],
        [-0.7128,  2.1491, -0.8216]])
ouput:tensor([[-2.2594,  0.8974, -0.0000],
        [-1.4257,  0.0000, -1.6432]])


##普通Dropout
对每一个元素独立进行随机置零或缩放。


##DropoutNd
按channel删除而不是按每个元素删除。

Dropout1d：用于一维数据场景，随机删除整个channel。

Dropout2d：用于二维数据场景，随机删除整个feature map。

Dropout3d：用于三维数据场景，随即删除整个3D feature map。


In [9]:
import torch
import torch.nn as nn
input = torch.randn(2,2,2,2)
dropout = nn.Dropout2d(p=0.5)
output = dropout(input)
print(f"input:{input}")
print(f"ouput:{output}")

input:tensor([[[[ 0.0680, -0.4801],
          [ 0.7840, -0.2442]],

         [[-0.2367, -1.9860],
          [-1.3584, -0.7947]]],


        [[[-0.1290, -0.5291],
          [-1.5907, -0.1151]],

         [[-1.9917,  1.6181],
          [-1.7902, -0.5402]]]])
ouput:tensor([[[[ 0.0000, -0.0000],
          [ 0.0000, -0.0000]],

         [[-0.4733, -3.9721],
          [-2.7167, -1.5894]]],


        [[[-0.2581, -1.0582],
          [-3.1815, -0.2302]],

         [[-3.9834,  3.2362],
          [-3.5804, -1.0803]]]])


##AlphaDropout
保持数据均值约为0，方差约为1的统计特性，常用于在SELU激活函数的后面。

AlphaDropout 会用一个特殊值替代被 dropout 的元素，并进行缩放，从而尽量维持理想的数学统计特性。

In [12]:
import torch
import torch.nn as nn
input = torch.randn(2,2,2)
dropout = nn.AlphaDropout(p=0.1)
output = dropout(input)
print(f"input:{input}")
print(f"ouput:{output}")

input:tensor([[[ 0.3585,  0.5426],
         [ 0.9018, -1.2154]],

        [[ 0.0206,  1.3321],
         [ 1.7153, -0.3668]]])
ouput:tensor([[[ 0.4923,  0.6619],
         [ 0.9928, -1.4577]],

        [[ 0.1809,  1.3892],
         [ 1.7423, -0.1759]]])
